In [ ]:
# !pip install -U huggingface_hub openai tapipy

In [ ]:
import getpass
from tapipy.tapis import Tapis
from pathlib import Path
import time
import json


tenant = 'public'
environment = ''
base_url = f'https://{tenant}{environment}.tapis.io' 

# Enter Tapis Username. Example: trainingXX
tapis_username = input('Username: ')
# Enter Tapis password. Example: trainingXX
tapis_password = getpass.getpass(prompt='Password: ', stream=None)


#Create python Tapis client for user
client = Tapis(base_url= base_url, username=tapis_username, password=tapis_password)
# *** Tapis v3: Call to Tokens API
client.get_tokens()
# Print Tapis v3 token
client.access_token

In [ ]:
nairrJobDefJSON = """
{
  "name": "tap_flexserv_vista_test",
  "appId": "FlexServ-vista-nairr",
  "appVersion": "1.4.0",
  "execSystemId": "vista-test-nairr",
  "tenant": "public",
  "execSystemLogicalQueue": "gh",
  "maxMinutes": 60,
  "parameterSet": {
    "appArgs": [
      {
        "arg": "--flexserv-port 8000",
        "name": "flexServPort"
      },
      {
        "arg": "--model-name Qwen/Qwen2.5-Coder-0.5B",
        "name": "modelName"
      },
      {
        "arg": "--enable-https",
        "name": "enableHttps"
      },
      {
        "arg": "--device auto",
        "name": "device"
      },
      {
        "arg": "--dtype bfloat16",
        "name": "dtype"
      },
      {
        "arg": "--attn-implementation sdpa",
        "name": "attnImplementation"
      },
      {
        "arg": "--model-timeout 86400",
        "name": "modelTimeout"
      },
      {
        "arg": "--quantization none",
        "name": "quantization"
      },
      {
        "arg": "--is-distributed 0",
        "name": "isDistributed"
      }
    ],
    "envVariables": [
      {
        "key": "PRI_MODEL_HOST",
        "value": "HOST_EVAL($SCRATCH)/flexserv/models"
      },
      {
        "key": "PUB_MODEL_HOST",
        "value": "/work/projects/aci/cic/models"
      }
    ],
    "schedulerOptions": [
      {
        "arg": "--tapis-profile tacc-apptainer",
        "name": "TACC Scheduler Profile"
      },
      {
        "name": "TACC Allocation",
        "description": "The TACC Allocation associated with this job execution",
        "include": true,
        "arg": "-A TACC-ACI-CIC"
      }
    ]
  },
  "visible": true
}
"""

nairrJobDef = json.loads(nairrJobDefJSON)
submitted_job = client.jobs.submitJob(**nairrJobDef)
print(f"Submitted job with UUID: {submitted_job.uuid}")

In [ ]:
submitted_job = client.jobs.getJob(jobUuid=submitted_job.uuid)

print(submitted_job.uuid)
print(submitted_job.name)
print(submitted_job.status)
print(submitted_job.execSystemId)
print(submitted_job.execSystemOutputDir)

In [ ]:
client.jobs.getJobHistory(jobUuid=submitted_job.uuid)

In [ ]:
accessInfoMarker = "ACCESS INFORMATION"
smokeTestMarker = "[smoke_test]"
serverReadyMarker = "Server ready to accept requests"

page=1
while page <= 5:
    output_page = client.files.getContents(systemId=submitted_job.execSystemId, path=submitted_job.execSystemOutputDir+"/tapisjob.out", more=page)
    if not output_page:
        break
    if isinstance(output_page, (bytes, bytearray)):
        log_text = output_page.decode("utf-8", errors="replace")
    else:
        log_text = str(output_page)
    if serverReadyMarker in log_text:
        # print(log_text, end="", flush=True)
        break
    time.sleep(5)
    page+=1

print("Useful Information from Job Logs:")
print("======================================================================")
lines = log_text.splitlines()

for i, line in enumerate(lines):
    if accessInfoMarker in line:
        print(f"{line}")
        end = min(i + 1 + 10, len(lines))
        for j in range(i + 1, end):
            print(lines[j])
    if smokeTestMarker in line:
        print(f"{line}")
    if serverReadyMarker in line:
        print(f"{line}")
        break

In [ ]:
# call OpenAI API to test if the inference server is working.
from openai import OpenAI
import logging

import httpx

# Create a custom httpx client with SSL verification disabled
http_client = httpx.Client(verify=False)


# Configuration from environment variables
api_key = FLEXSERV_KEY
base_url = f"https://{system_name}.tacc.utexas.edu:{FLEXSERV_LOGIN_PORT}/v1"

print(f"API Key: {api_key}")
print(f"Base URL: {base_url}")

client = OpenAI(
    api_key=api_key,
    base_url=base_url,
    http_client=http_client,
)

print("✓ OpenAI client initialized")

# Enable detailed logging for httpx (used by OpenAI client)
logging.basicConfig(level=logging.DEBUG)
httpx_logger = logging.getLogger("httpx")
httpx_logger.setLevel(logging.DEBUG)

# Or just enable OpenAI SDK logging
openai_logger = logging.getLogger("openai")
openai_logger.setLevel(logging.DEBUG)

print("✓ Request logging enabled - will show HTTP details below")

start_time = time.time()

# model_to_test = "Qwen/Qwen2.5-Coder-0.5B"
# model_to_test = "Qwen/Qwen2.5-Coder-7B-Instruct"
model_to_test = "Qwen/Qwen2.5-Coder-32B-Instruct"

# Test chat/completions endpoint with proper message format using openai.chat.completions.create
stream = client.chat.completions.create(
    model=model_to_test,
    messages=[
        {
            "role": "system",
            "content": "You are a coding assistant that writes clear, correct, beginner-friendly Python code for machine learning tasks.",
        },
        {
            "role": "user",
            "content": (
                "Write Python code that reads all images from an input directory stored in the variable IMAGE_DIR.\n\n"
                "TASK DESCRIPTION:\n"
                "- This is an IMAGE-LEVEL BINARY CLASSIFICATION task implemented using an object detection model.\n"
                "- The goal is to determine whether an image contains an animal or not.\n\n"
                "DATASET STRUCTURE:\n"
                "- IMAGE_DIR contains two subdirectories:\n"
                "  * animal     → images that contain at least one animal\n"
                "  * no_animal  → images that contain no animals\n"
                "- Ground-truth labels come ONLY from the directory name.\n\n"
                "MODEL REQUIREMENTS:\n"
                "- Use ONLY a pretrained Ultralytics YOLO detection model (no training or fine-tuning).\n"
                "- Load the model using the Ultralytics YOLO API.\n"
                "- Assume YOLO detects animals using a single class ID (animal).\n\n"
                "DETECTION LOGIC (IMPORTANT):\n"
                "- Run object detection on each image.\n"
                "- If the model produces AT LEAST ONE detection of class `animal` with confidence >= 0.5:\n"
                "    → The image-level prediction is `animal`.\n"
                "- If the model produces NO detections with confidence >= 0.5:\n"
                "    → The image-level prediction is `no_animal`.\n"
                "- YOLO never explicitly predicts `no_animal`; it is inferred from the absence of detections.\n\n"
                "EVALUATION METRICS:\n"
                "- Compare the image-level prediction with the ground-truth label.\n"
                "- Count:\n"
                "  * True Positives  (animal image, animal detected)\n"
                "  * True Negatives  (no_animal image, no detections)\n"
                "  * False Positives (no_animal image, detection present)\n"
                "  * False Negatives (animal image, no detections)\n\n"
                "ACCURACY DEFINITION:\n"
                "- Overall accuracy = (True Positives + True Negatives) / Total Images\n\n"
                "OUTPUT REQUIREMENTS:\n"
                "- Print for each image: filename, detected classes, and confidence scores.\n"
                "- At the end, print:\n"
                "  * Total images processed\n"
                "  * Total animal images (ground truth)\n"
                "  * True Positives\n"
                "  * True Negatives\n"
                "  * False Positives\n"
                "  * False Negatives\n"
                "  * Overall detection accuracy\n\n"
                "CODING REQUIREMENTS:\n"
                "- Store the directory path in IMAGE_DIR.\n"
                "- Read only .jpg, .jpeg, and .png files.\n"
                "- Include all required Python packages.\n"
                "- Include clear comments explaining each step.\n"
                "- Keep the code simple, readable, and beginner-friendly.\n\n"
                "After the code, briefly explain how the program works in plain English."
            ),
        },
    ],
    stream=True,
    max_tokens=1000,
    n=1,
    temperature=0.1,
    # Add other parameters as needed
    # stop=None,
    # top_p=1.0,
    # presence_penalty=0,
    # frequency_penalty=0,
    # logit_bias=None,
    # user=None
)

end_time = time.time()
elapsed_time = end_time - start_time
for chunk in stream:
    if chunk.choices[0].delta.content is not None:
        print(chunk.choices[0].delta.content, end="", flush=True)

print("\n\n✓ Streaming complete")

In [ ]:
client.jobs.cancelJob(jobUuid=submitted_job.uuid)